# 📑 [제출 3] 구독 해지 예측 모델 고도화 및 최적화 보고서

본 프로젝트는 서비스 이탈 가능성이 높은 사용자를 선제적으로 탐지하여 방어하기 위한 머신러닝 모델링 과정을 담고 있습니다.

## 🎯 핵심 목표
- **Recall(Class 0) 극대화**: 이탈 후보군을 80% 이상 정확히 식별
- **단계적 개선**: 1~3차에 걸친 하이퍼파라미터 및 샘플링 전략 최적화
- **비즈니스 가치**: 탐지된 이탈 후보군에 대한 맞춤형 마케팅 기반 마련

---
## 📝 모델 최적화 히스토리 요약
| 단계 | 최적화 전략 | 핵심 파라미터 | 성과 (Recall) |
| :--- | :--- | :--- | :---: |
| **Baseline** | CatBoost 기본 모델 | Default | 0.6833 |
| **1차 튜닝** | 기초 파라미터 최적화 | `depth: 5, lr: 0.018` | 0.7500 |
| **2차 튜닝** | 규제 강화 및 샘플링 최적화 | `l2_leaf_reg: 1.46` | 0.7853 |
| **3차 튜닝** | **가중치 강화 및 정밀 최적화** | `depth: 7, l2_reg: 13.5` | **0.8060** |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    recall_score, confusion_matrix, classification_report, 
    accuracy_score, f1_score, roc_auc_score
)

from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier

# 한글 폰트 및 시각화 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_palette("husl")

def plot_custom_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    recall = recall_score(y_true, y_pred, pos_label=0)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', 
                xticklabels=['해지(0)', '유지(1)'], 
                yticklabels=['해지(0)', '유지(1)'])
    plt.title(f'{title}\n[Recall(Churn): {recall:.4f}]', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted', fontsize=10)
    plt.ylabel('Actual', fontsize=10)
    plt.show()

## 1. 데이터 로드 및 고도화된 피처 엔지니어링
`mock_data_3.csv`를 활용하여 사용자의 실질적 비용 부담과 이탈 의도를 반영하는 도메인 변수를 생성합니다.

In [ ]:
def advanced_feature_engineering(data):
    df = data.copy()
    
    # 1. 수치형 매핑
    use_freq_map = {"rare": 1, "monthly": 2, "weekly": 3, "frequent": 4}
    recency_map = {">30d": 1, "7-30d": 2, "1-7d": 3, "<1d": 4}
    df["use_freq_idx"] = df["use_frequency"].map(use_freq_map)
    df["recency_idx"] = df["last_use_recency"].map(recency_map)
    
    # 2. 실질 비용 부담 (Effective Cost)
    df["effective_cost"] = (df["monthly_cost"] - df["discount_amount"]).clip(lower=0)
    
    # 3. 가치 격차 (Value Gap) - 높을수록 유지 의향 높음
    # 공식: (빈도 + 최근성 + 필요도) - 비용 부담
    df["value_gap"] = (df["use_freq_idx"] + df["recency_idx"] + df["perceived_necessity"] - df["cost_burden"])
    
    # 4. 이탈 신호 (Churn Signal)
    # 저빈도/저활용 및 고비용 부담자 식별
    df["churn_signal"] = ((df["use_freq_idx"] <= 2) & (df["recency_idx"] <= 2) & (df["cost_burden"] >= 4)).astype(int)
    
    # 5. 상호작용 변수
    df["necessity_recency_inter"] = df["perceived_necessity"] * df["recency_idx"]
    df["cost_per_frequency"] = df["effective_cost"] / (df["use_freq_idx"] + 1)
    
    return df

# 데이터 로드 및 실행
df_raw = pd.read_csv("mock_data_3.csv")
df_p = advanced_feature_engineering(df_raw)

# 피처 정의
features = [
    "subscription_type", "effective_cost", "perceived_necessity", 
    "cost_burden", "would_rebuy", "replacement_available", 
    "billing_cycle", "value_gap", "churn_signal", 
    "necessity_recency_inter", "cost_per_frequency"
]
target = "target"

# 분할
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_p[features], df_p[target], test_size=0.2, random_state=42, stratify=df_p[target]
)

# 전처리기 정의
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["subscription_type"]),
    ("num", StandardScaler(), [f for f in features if f != "subscription_type"])
])

X_train_pre = preprocessor.fit_transform(X_train_raw)
X_test_pre = preprocessor.transform(X_test_raw)

print(f"✅ 데이터 전처리 및 피처 생성 완료: {X_train_pre.shape}")

## 2. 단계별 모델 개선 (Phase 1 & 2)
1차 및 2차 튜닝을 통한 성능 개선 추이를 확인합니다.

In [ ]:
# [Round 1] 초기 최적화
sm1 = SMOTE(random_state=42)
X_r1, y_r1 = sm1.fit_resample(X_train_pre, y_train)
model1 = CatBoostClassifier(iterations=432, learning_rate=0.018, depth=5, verbose=0, random_seed=42)
model1.fit(X_r1, y_r1)
y_p1 = np.where(model1.predict_proba(X_test_pre)[:, 0] >= 0.4, 0, 1)

# [Round 2] 규제 및 샘플링 고도화
sm2 = SMOTE(sampling_strategy=0.63, random_state=42)
X_r2, y_r2 = sm2.fit_resample(X_train_pre, y_train)
model2 = CatBoostClassifier(iterations=717, learning_rate=0.0501, depth=5, l2_leaf_reg=1.46, verbose=0, random_seed=42)
model2.fit(X_r2, y_r2)
y_p2 = np.where(model2.predict_proba(X_test_pre)[:, 0] >= 0.3, 0, 1)

# 결과 시각화
print("=== 1차 최적화 결과 ===")
plot_custom_cm(y_test, y_p1, "Phase 1: Initial Tuning")
print("=== 2차 최적화 결과 ===")
plot_custom_cm(y_test, y_p2, "Phase 2: Regularized Tuning")

## 3. 최종 최적화 (Phase 3: Recall Maximization)
가장 높은 이탈 탐지율을 달성한 최종 모델 설정입니다.
- **특징**: 해지군(Class 0)에 높은 페널티를 부여하는 가중치 학습 적용

In [ ]:
# 최종 3차 튜닝 파라미터 적용
best_params = {
    "iterations": 361,
    "learning_rate": 0.0395,
    "depth": 7,
    "l2_leaf_reg": 13.53,
    "random_strength": 5.73,
    "bagging_temperature": 0.47,
    "border_count": 175,
    "min_data_in_leaf": 19,
    "random_seed": 42,
    "verbose": 0,
    "loss_function": "Logloss"
}

# SMOTE 최적 비율 적용 (1:1 근접)
sm_final = SMOTE(sampling_strategy=0.9994, random_state=42)
X_final, y_final = sm_final.fit_resample(X_train_pre, y_train)

# 최종 모델 학습
final_model = CatBoostClassifier(**best_params)
final_model.fit(X_final, y_final)

# 최종 임계값(0.34) 적용 및 예측
threshold = 0.34
y_prob_final = final_model.predict_proba(X_test_pre)[:, 0]
y_pred_final = np.where(y_prob_final >= threshold, 0, 1)

# 최종 평가 및 시각화
print("🚀 [최종] 3차 최적화 모델 평가 보고서")
print(classification_report(y_test, y_pred_final, target_names=['해지(0)', '유지(1)']))
plot_custom_cm(y_test, y_pred_final, "Phase 3: Final Optimized Model")

## 4. 모델 분석: 어떤 요인이 해지를 유발하는가?

In [ ]:
# 피처 중요도 추출
fi = final_model.get_feature_importance()
ohe_cols = list(preprocessor.named_transformers_['cat'].get_feature_names_out(["subscription_type"]))
all_cols = ohe_cols + [f for f in features if f != "subscription_type"]

fi_df = pd.DataFrame({'Feature': all_cols, 'Importance': fi}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis')
plt.title('Final Model Feature Importance Analysis', fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 🏁 5. 결론 및 비즈니스 제언
1. **탐지 성능**: 최종 모델은 **Recall 80.6%**를 달성하여, Baseline 대비 해지 잠재 고객을 약 12% 더 많이 포착합니다.
2. **핵심 요인**: 피처 분석 결과 `value_gap`과 `churn_signal`이 이탈 예측에 가장 결정적인 역할을 하는 것으로 나타났습니다.
3. **제언**: 예측 확률 0.34 이상의 사용자에게 공격적인 리텐션 캠페인(구독 갱신 할인 등)을 집행할 것을 권장합니다.